# 📘 Activity 09: Logistic Regression with scikit-learn + StratifiedKFold

**Track:** Logistic Regression  
**Level:** Intermediate  
**K-Fold:** ✅ StratifiedKFold — class imbalance handling

---

## 📖 What You Will Learn
- Logistic regression for binary and multi-class classification
- The sigmoid function and decision boundary
- Probability outputs vs hard decisions
- **StratifiedKFold** — why regular KFold fails on imbalanced classes
- Hyperparameter C (inverse regularisation strength)
- ROC-AUC, confusion matrix, classification report

---

## 🧠 Concept: Logistic Regression

Logistic regression predicts the **probability** of class 1:

$$P(y=1|x) = \sigma(\beta_0 + \beta_1 x_1 + \ldots) = \frac{1}{1 + e^{-z}}$$

**Loss:** Binary Cross-Entropy (Log Loss)
$$\mathcal{L} = -\frac{1}{n}\sum [y_i \log(\hat{p}_i) + (1-y_i)\log(1-\hat{p}_i)]$$

> Unlike linear regression (MSE), logistic regression uses log loss because MSE creates
> a non-convex loss surface for classification — gradient descent can get stuck.

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.datasets import load_breast_cancer, make_classification
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import (
    train_test_split, StratifiedKFold, KFold,
    cross_val_score, cross_validate, GridSearchCV
)
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import Pipeline
from sklearn.metrics import (
    accuracy_score, precision_score, recall_score, f1_score,
    roc_auc_score, confusion_matrix, classification_report,
    roc_curve, RocCurveDisplay
)

import warnings; warnings.filterwarnings('ignore')
np.random.seed(42)
print('✅ Ready')

## 📊 Dataset: Breast Cancer (Binary Classification)

In [ ]:
data = load_breast_cancer(as_frame=True)
X = data.data
y = data.target    # 0 = malignant, 1 = benign

print(f'Samples: {X.shape[0]}  Features: {X.shape[1]}')
print(f'Class distribution: {dict(y.value_counts().sort_index())}')
print(f'Class balance: {y.value_counts(normalize=True).round(3).to_dict()}')

# Visualise class balance
plt.figure(figsize=(5, 3))
y.value_counts().plot(kind='bar', color=['tomato', 'steelblue'], edgecolor='white')
plt.xticks([0, 1], ['Malignant (0)', 'Benign (1)'], rotation=0)
plt.title('Class Distribution')
plt.ylabel('Count')
plt.tight_layout()
plt.show()

## 🔧 Step 1 — Baseline Model (Single Split)

In [ ]:
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y   # stratify maintains class proportions!
)

pipe = Pipeline([
    ('scaler', StandardScaler()),
    ('lr',     LogisticRegression(max_iter=1000, random_state=42))
])

pipe.fit(X_train, y_train)
y_pred  = pipe.predict(X_test)
y_proba = pipe.predict_proba(X_test)[:, 1]   # P(class=1)

print('=== Single Split Results ===')
print(classification_report(y_test, y_pred, target_names=['Malignant', 'Benign']))
print(f'ROC-AUC: {roc_auc_score(y_test, y_proba):.4f}')

## 🔄 Step 2 — StratifiedKFold vs Regular KFold

### 🧠 Concept: Why StratifiedKFold?

With **class imbalance**, regular KFold may produce folds where one class is severely
under-represented or even missing — the model trains on unbalanced data and metrics are misleading.

**StratifiedKFold** ensures each fold has the **same class proportion** as the original dataset.

Example with 10 % minority class:

| Fold | Regular KFold (minority %) | StratifiedKFold (minority %) |
|---|---|---|
| 1 | 3 % | 10 % |
| 2 | 18 % | 10 % |
| 3 | 5 % | 10 % |
| ... | Varies wildly | Always ~10 % |

> **Rule:** Always use StratifiedKFold for classification. Always.

In [ ]:
# Create a more imbalanced dataset to demonstrate the contrast
X_imb, y_imb = make_classification(
    n_samples=500,
    n_features=20,
    weights=[0.9, 0.1],    # 90% class 0, 10% class 1
    random_state=42
)

kf_regular    = KFold(n_splits=5, shuffle=True, random_state=42)
kf_stratified = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)

print('Regular KFold — class 1 % per fold:')
for fold, (tr_idx, te_idx) in enumerate(kf_regular.split(X_imb, y_imb), 1):
    class1_pct = y_imb[te_idx].mean() * 100
    print(f'  Fold {fold}: {class1_pct:.1f}%')

print('\nStratifiedKFold — class 1 % per fold:')
for fold, (tr_idx, te_idx) in enumerate(kf_stratified.split(X_imb, y_imb), 1):
    class1_pct = y_imb[te_idx].mean() * 100
    print(f'  Fold {fold}: {class1_pct:.1f}%')

In [ ]:
# ── StratifiedKFold on breast cancer ──────────────────────────────────────────
skf = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)

metrics_per_fold = []

for fold, (tr_idx, te_idx) in enumerate(skf.split(X, y), 1):
    X_tr, X_te = X.iloc[tr_idx], X.iloc[te_idx]
    y_tr, y_te = y.iloc[tr_idx], y.iloc[te_idx]

    pipe.fit(X_tr, y_tr)
    y_pred  = pipe.predict(X_te)
    y_proba = pipe.predict_proba(X_te)[:, 1]

    metrics_per_fold.append({
        'Fold':      fold,
        'Accuracy':  accuracy_score(y_te, y_pred),
        'Precision': precision_score(y_te, y_pred),
        'Recall':    recall_score(y_te, y_pred),
        'F1':        f1_score(y_te, y_pred),
        'ROC-AUC':   roc_auc_score(y_te, y_proba),
    })

fold_df = pd.DataFrame(metrics_per_fold)
print(fold_df.round(4).to_string(index=False))
print('\nMean performance:')
print(fold_df.drop(columns='Fold').mean().round(4))

## 🎚️ Step 3 — Hyperparameter Tuning: C (Regularisation Strength)

In [ ]:
# WHY C? C = 1/α. Small C = strong regularisation. Large C = weak regularisation.
# We use GridSearchCV which internally uses StratifiedKFold for classification.

param_grid = {'lr__C': np.logspace(-4, 4, 20)}

grid = GridSearchCV(
    pipe,
    param_grid,
    cv=skf,
    scoring='roc_auc',
    return_train_score=True,
    n_jobs=-1
)
grid.fit(X, y)

best_C  = grid.best_params_['lr__C']
best_cv = grid.best_score_
print(f'Best C: {best_C:.5f}  CV ROC-AUC: {best_cv:.4f}')

In [ ]:
# Validation curve
results = pd.DataFrame(grid.cv_results_)

plt.figure(figsize=(10, 5))
plt.semilogx(results['param_lr__C'], results['mean_train_score'],
             'b-o', markersize=4, label='Train ROC-AUC')
plt.semilogx(results['param_lr__C'], results['mean_test_score'],
             'r-s', markersize=4, label='CV ROC-AUC')
plt.fill_between(results['param_lr__C'],
                  results['mean_test_score'] - results['std_test_score'],
                  results['mean_test_score'] + results['std_test_score'],
                  alpha=0.2, color='red')
plt.axvline(best_C, color='green', linestyle='--', label=f'Best C={best_C:.4f}')
plt.xlabel('C (log scale)')
plt.ylabel('ROC-AUC')
plt.title('Validation Curve — Logistic Regression C Parameter')
plt.legend()
plt.tight_layout()
plt.show()

## 📈 Step 4 — ROC Curve & Confusion Matrix

In [ ]:
# Refit best model
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)
best_model = grid.best_estimator_
best_model.fit(X_train, y_train)

y_pred  = best_model.predict(X_test)
y_proba = best_model.predict_proba(X_test)[:, 1]

fig, axes = plt.subplots(1, 2, figsize=(13, 5))

# Confusion matrix
cm = confusion_matrix(y_test, y_pred)
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues',
            xticklabels=['Malignant', 'Benign'],
            yticklabels=['Malignant', 'Benign'],
            ax=axes[0])
axes[0].set_title('Confusion Matrix')
axes[0].set_ylabel('Actual')
axes[0].set_xlabel('Predicted')

# ROC curve
RocCurveDisplay.from_predictions(
    y_test, y_proba,
    name='Logistic Regression',
    ax=axes[1]
)
axes[1].set_title('ROC Curve')

plt.tight_layout()
plt.show()

print(f'\nTest ROC-AUC: {roc_auc_score(y_test, y_proba):.4f}')
print(f'Test F1:      {f1_score(y_test, y_pred):.4f}')

## 🎯 Summary

| Concept | Detail |
|---|---|
| StratifiedKFold | Preserves class proportions per fold — mandatory for classification |
| C parameter | Inverse regularisation: small C → strong penalty |
| ROC-AUC | Threshold-independent measure of discriminative ability |
| `stratify=y` in train_test_split | Also maintains class proportion in the holdout split |

---

**Next:** [10_logistic_regression_from_scratch.ipynb](10_logistic_regression_from_scratch.ipynb)